In [ ]:
# AT4-KN01 (spec v1.2 "Agent test 4 (scaled): unseen documents with gold answers, design before running", 2026-11-20):
# the thorough-mode index (Docling parse/OCR, method1_tree T384 chunks, bge-m3 vectors) of the 100 agent-test-4 PDFs
# on a Kaggle GPU, because the laptop estimate is over 3 hours. No answers here (no API keys on Kaggle): unzip the
# output at the repo root, then run the batch command on the laptop, where every index is a cache hit.
# Needs: GPU T4, Internet on (clone + model download), dataset AT4-KD01 attached (data/agent_test4/questions.csv and
# data/agent_test4/pdfs/*.pdf). No secrets. These documents are in no split; none is a study document.
COMMIT = "a154b9f"  # the commit that holds src/agent/test4.py and pdfs/agent_test4_picks.csv; set before running
assert COMMIT, "set COMMIT to the commit with the agent test 4 code and picks"
!git clone -q https://github.com/amanchaurasia512/Self_optimized_Agent_Ai_chunking.git /kaggle/working/repo
!cd /kaggle/working/repo && git checkout -q {COMMIT} && git log --oneline -1 && git status --porcelain | head

In [ ]:
# Installs, pins read from the clone (no versions typed here): the Docling environment (docs/constraints-app.txt, the
# pip freeze of .venv-docling, where the agent runs) for Docling, its OCR and every package the index step imports;
# requirements.txt only for a name that file lacks. The same file, without Kaggle's own CUDA torch/torchvision and
# numpy/pandas/pyarrow/scipy (kept as in rob-kn01) and OpenCV (Kaggle ships opencv-python, -contrib and -headless
# 4.14; rapidocr needs headless >= 4.5.1.48, and a second cv2 build would mix files), constrains every dependency. The cache is reused on
# the laptop only if the Docling version matches (src/agent/index.cached_index).
import re
from pathlib import Path

REPO, WORK = Path("/kaggle/working/repo"), Path("/kaggle/working")
KAGGLE_PROVIDED = {"torch", "torchvision", "numpy", "pandas", "pyarrow", "scipy", "opencv-python-headless"}
WANTED = ["docling", "docling-slim", "docling-core", "docling-parse", "docling-ibm-models", "rapidocr",
          "sentence-transformers", "transformers", "tokenizers", "huggingface-hub", "stanza",
          "langchain-classic", "langchain-community", "langchain-core", "langchain-experimental", "langchain-protocol",
          "langchain-text-splitters", "google-genai", "pdfplumber", "pdfminer-six", "pypdf", "pypdfium2", "jsonschema",
          "typer", "psutil", "httpx", "pydantic"]
name = lambda line: re.split(r"==", line)[0].strip().lower().replace("_", "-").replace(".", "-")
pins = lambda file: {name(line): line.strip() for line in (REPO / file).read_text().splitlines()
                     if "==" in line and not line.lstrip().startswith("#")}
app, study = pins("docs/constraints-app.txt"), pins("requirements.txt")
missing = [package for package in WANTED if package not in app and package not in study]
assert not missing, f"no pin in docs/constraints-app.txt or requirements.txt: {missing}"
(WORK / "at4_pins.txt").write_text("\n".join(app.get(package) or study[package] for package in WANTED) + "\n")
(WORK / "at4_constraints.txt").write_text("\n".join(line for package, line in app.items() if package not in KAGGLE_PROVIDED) + "\n")
print((WORK / "at4_pins.txt").read_text())
!pip install -q -r /kaggle/working/at4_pins.txt -c /kaggle/working/at4_constraints.txt
!pip freeze > /kaggle/working/at4-kn01_pip_freeze.txt

In [ ]:
# Fail fast, before any heavy work: the index step's imports, the Docling converter loaded once (layout, table and OCR
# models), Docling 2.143.0, the GPU, and no answer-provider or Gemini key on this machine (the index step makes no
# API calls; src/agent/providers.py and src/chunk/llm.py read keys only when answering).
!grep -i -E "^(docling|torch|torchvision|transformers|sentence-transformers|tokenizers|stanza|numpy|pandas)==" /kaggle/working/at4-kn01_pip_freeze.txt
!cd /kaggle/working/repo && python -c "import src.agent.test4, src.chunk.chunkers, src.parse.docling_parser, src.agent.ask, src.agent.batch; print('imports OK')"
import sys
from importlib.metadata import version

sys.path.insert(0, str(REPO))
import src.agent.test4, src.chunk.chunkers, src.parse.docling_parser
from docling.datamodel.base_models import InputFormat

converter, parser_info = src.parse.docling_parser.make_converter()
converter.initialize_pipeline(InputFormat.PDF)
del converter
assert version("docling") == "2.143.0", f"Docling {version('docling')}, not 2.143.0"
print("imports OK | Docling", version("docling"), "| converter loaded |", parser_info["settings"])
assert not (Path.home() / ".config" / "pdf-agent").exists() and not (Path.home() / ".config" / "gemini").exists(), "keys found"
import torch
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "turn the GPU on"

In [ ]:
# Copy AT4-KD01 into the clone at its repo paths. Every PDF must be in pdfs/agent_test4_picks.csv of this commit
# with the same SHA-256, and questions.csv must name exactly those PDFs; anything else stops the notebook.
import hashlib
import shutil
import zipfile

import pandas as pd

# Kaggle mounts aman125/at4-kd01 at /kaggle/input/datasets/aman125/at4-kd01 (older notebooks: /kaggle/input/at4-kd01).
# Expected inside: questions.csv and pdfs/*.pdf (data/agent_test4/ uploaded without its cache), or pdfs as a zip.
CANDIDATES = [Path("/kaggle/input/datasets/aman125/at4-kd01"), Path("/kaggle/input/at4-kd01")]
DATASET = next((path for path in CANDIDATES if path.is_dir()), None)
assert DATASET, f"attach the dataset aman125/at4-kd01; not found at {[str(path) for path in CANDIDATES]}"
for archive in DATASET.rglob("*.zip"):  # uploaded with --dir-mode zip and not unpacked by Kaggle
    zipfile.ZipFile(archive).extractall(Path("/kaggle/temp/at4-kd01") / archive.stem)
roots = [DATASET, Path("/kaggle/temp/at4-kd01")]
questions_file = next(path for root in roots if root.is_dir() for path in root.rglob("questions.csv"))
found = {path.name: path for root in roots if root.is_dir() for path in root.rglob("*.pdf")}
picks = pd.read_csv(REPO / "pdfs" / "agent_test4_picks.csv")
questions = pd.read_csv(questions_file)
assert set(questions["pdf"]) == set(picks["pdf"]) and len(picks) == 100, "questions.csv and the picks differ"
sha = lambda path: hashlib.sha256(path.read_bytes()).hexdigest()
for pick in picks.itertuples(index=False):
    source, target = found[Path(pick.pdf).name], REPO / pick.pdf
    assert sha(source) == pick.sha256, pick.pdf
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(source, target)
shutil.copyfile(questions_file, REPO / "data" / "agent_test4" / "questions.csv")
print(len(picks), "PDFs copied and checked |", picks.groupby("kind")["pages"].sum().to_dict(), "pages")

In [ ]:
# Every index, one document at a time; failures are recorded in the run record and left out of batch_questions.csv.
!cd /kaggle/working/repo && python -m src.agent.test4 index --cache-root data/agent_test4/cache --machine kaggle --source AT4-KN01

In [ ]:
# One zip of the cache, batch_questions.csv and the run record, at repo paths (unzip at the repo root on the laptop).
files = sorted(path for path in (REPO / "data" / "agent_test4" / "cache").rglob("*") if path.is_file())
files += [REPO / "data" / "agent_test4" / "batch_questions.csv"] + sorted((REPO / "runs").glob("A-AT4-KN01-*.json"))
with zipfile.ZipFile("/kaggle/working/at4-kn01_outputs.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for path in files:
        archive.write(path, path.relative_to(REPO))
    archive.write("/kaggle/working/at4-kn01_pip_freeze.txt", "data/external/kaggle_out/AT4-KN01/at4-kn01_pip_freeze.txt")
print(len(files), "files zipped | run records:", sorted(path.name for path in files if path.parent.name == "runs"))
!ls -la /kaggle/working